# Notebook 03: KNN, Decision Trees & Ensemble Methods

**Capstone Stage 1 | Modules 8, 9, 10**  
**Dataset:** Polish Companies Bankruptcy (UCI ID 365)  
**Author:** Srini | Imperial College London — Professional Certificate in ML & AI

---

## Objectives

- Apply **K-Nearest Neighbours** with hyperparameter tuning of k (Module 8)
- Build **Decision Trees** with pruning — Gini vs entropy, stopping criteria (Module 9)
- Implement **ensemble methods**: Bagging (Random Forest) and Boosting (Gradient Boosting) (Module 10)
- Compare all models on the same evaluation framework (AUC, F1)
- Demonstrate how ensemble methods outperform single models on this credit dataset

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
np.random.seed(42)

# Load and prepare data
df = pd.read_csv('../data/polish_bankruptcy.csv')
feature_cols = [c for c in df.columns if c.startswith('X')]
X_raw = df[feature_cols].values
y = df['target'].values

X_trainval, X_test, y_trainval, y_test = train_test_split(X_raw, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

preprocessor = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())])
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_pp, y_train)

print('Data ready. Train (SMOTE):', X_train_sm.shape, '| Val:', X_val_pp.shape)

---
## 1. K-Nearest Neighbours — Hyperparameter Tuning (Module 8)

In [ ]:
# KNN: tune k and measure AUC — classic bias-variance via k
k_values = [1, 3, 5, 7, 11, 15, 21, 31, 51]
knn_train_auc, knn_val_auc = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    knn.fit(X_train_sm, y_train_sm)
    knn_train_auc.append(roc_auc_score(y_train_sm, knn.predict_proba(X_train_sm)[:,1]))
    knn_val_auc.append(roc_auc_score(y_val, knn.predict_proba(X_val_pp)[:,1]))

best_k = k_values[np.argmax(knn_val_auc)]
print(f'Best k (by validation AUC): k={best_k} → AUC={max(knn_val_auc):.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_values, knn_train_auc, 'o-', color='#2196F3', label='Training AUC', linewidth=2)
ax.plot(k_values, knn_val_auc,   's-', color='#F44336', label='Validation AUC', linewidth=2)
ax.axvline(best_k, color='green', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
ax.set_xlabel('k (number of neighbours)'); ax.set_ylabel('AUC-ROC')
ax.set_title('KNN Hyperparameter Tuning — k vs AUC (Polish Bankruptcy)', fontweight='bold')
ax.legend(); ax.set_xticks(k_values)
plt.tight_layout()
plt.savefig('../reports/03_knn_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservation: Small k overfits (high train, lower val AUC).')
print('Large k underfits. Optimal k is in the middle — classic bias-variance trade-off.')

---
## 2. Decision Trees — Gini vs Entropy, Pruning (Modules 9)

In [ ]:
# Compare Gini vs Entropy at various depths
results = []
for criterion in ['gini', 'entropy']:
    for depth in [3, 5, 7, 10, 15, None]:
        dt = DecisionTreeClassifier(criterion=criterion, max_depth=depth, random_state=42)
        dt.fit(X_train_sm, y_train_sm)
        results.append({
            'criterion': criterion, 'max_depth': depth if depth else 'None',
            'train_auc': roc_auc_score(y_train_sm, dt.predict_proba(X_train_sm)[:,1]),
            'val_auc':   roc_auc_score(y_val, dt.predict_proba(X_val_pp)[:,1]),
            'n_leaves':  dt.get_n_leaves()
        })
results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, criterion, colour in [(axes[0], 'gini', '#2196F3'), (axes[1], 'entropy', '#F44336')]:
    sub = results_df[results_df['criterion']==criterion].copy()
    sub['depth_num'] = range(len(sub))
    ax.plot(sub['depth_num'], sub['train_auc'], 'o-', color=colour, label='Train AUC')
    ax.plot(sub['depth_num'], sub['val_auc'],   's--', color=colour, alpha=0.6, label='Val AUC')
    ax.set_xticks(sub['depth_num'])
    ax.set_xticklabels(['D=3','D=5','D=7','D=10','D=15','Full'])
    ax.set_title(f'Decision Tree ({criterion.capitalize()}) — Depth vs AUC', fontweight='bold')
    ax.set_ylabel('AUC-ROC'); ax.set_ylim(0.5, 1.02); ax.legend()

plt.suptitle('Decision Tree: Gini vs Entropy Criterion at Varying Depths', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/03_decision_tree_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_dt_row = results_df.loc[results_df['val_auc'].idxmax()]
print(f'Best Decision Tree: criterion={best_dt_row["criterion"]}, depth={best_dt_row["max_depth"]}')
print(f'  Val AUC={best_dt_row["val_auc"]:.4f}, leaves={best_dt_row["n_leaves"]}')

# Train best DT and visualise (shallow version for interpretability)
dt_best = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_best.fit(X_train_sm, y_train_sm)

fig, ax = plt.subplots(figsize=(20, 8))
feature_names = [f'X{i+1}' for i in range(X_train_sm.shape[1])]
plot_tree(dt_best, ax=ax, feature_names=feature_names, class_names=['Solvent','Bankrupt'],
          filled=True, rounded=True, fontsize=8, max_depth=3)
ax.set_title('Decision Tree (depth=4) — Polish Bankruptcy Credit EWS', fontweight='bold', fontsize=12)
plt.savefig('../reports/03_decision_tree_plot.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Ensemble Methods — Bagging (Random Forest) & Boosting (Module 10)

In [ ]:
# Random Forest — Bagging ensemble
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_val_proba = rf.predict_proba(X_val_pp)[:,1]
rf_val_auc   = roc_auc_score(y_val, rf_val_proba)
rf_val_f1    = f1_score(y_val, (rf_val_proba>=0.5).astype(int))

# Gradient Boosting — Boosting ensemble
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                  subsample=0.8, random_state=42)
gb.fit(X_train_sm, y_train_sm)
gb_val_proba = gb.predict_proba(X_val_pp)[:,1]
gb_val_auc   = roc_auc_score(y_val, gb_val_proba)
gb_val_f1    = f1_score(y_val, (gb_val_proba>=0.5).astype(int))

print(f'Random Forest  (Bagging):   AUC={rf_val_auc:.4f}  F1={rf_val_f1:.4f}')
print(f'Gradient Boost (Boosting):  AUC={gb_val_auc:.4f}  F1={gb_val_f1:.4f}')

# ROC comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
lr.fit(X_train_sm, y_train_sm)
lr_proba = lr.predict_proba(X_val_pp)[:,1]
lr_auc = roc_auc_score(y_val, lr_proba)

knn_best = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn_best.fit(X_train_sm, y_train_sm)
knn_proba = knn_best.predict_proba(X_val_pp)[:,1]
knn_auc = roc_auc_score(y_val, knn_proba)

models = [
    ('Logistic Regression', lr_proba, lr_auc, '#9C27B0'),
    (f'KNN (k={best_k})', knn_proba, knn_auc, '#FF9800'),
    ('Decision Tree (D=4)', dt_best.predict_proba(X_val_pp)[:,1],
     roc_auc_score(y_val, dt_best.predict_proba(X_val_pp)[:,1]), '#795548'),
    ('Random Forest', rf_val_proba, rf_val_auc, '#2196F3'),
    ('Gradient Boosting', gb_val_proba, gb_val_auc, '#F44336'),
]

for name, proba, auc, colour in models:
    fpr, tpr, _ = roc_curve(y_val, proba)
    axes[0].plot(fpr, tpr, color=colour, linewidth=2, label=f'{name} ({auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves — Model Comparison', fontweight='bold')
axes[0].legend(fontsize=8)

# Feature importance — Random Forest
importances = rf.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]
with open('../data/feature_map.json') as f:
    fm = json.load(f)
top_labels = [fm.get(f'X{i+1}', f'X{i+1}').replace('_',' ')[:30] for i in top_idx]
axes[1].barh(range(15), importances[top_idx][::-1], color='#2196F3', alpha=0.8)
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([l for l in top_labels[::-1]], fontsize=8)
axes[1].set_xlabel('Feature Importance (Mean Decrease Impurity)')
axes[1].set_title('Random Forest — Top 15 Features', fontweight='bold')

plt.suptitle('Ensemble Methods: Random Forest & Gradient Boosting vs Baselines',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/03_ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Module': 13,
     'Val AUC': roc_auc_score(y_val, lr_proba),
     'Val F1': f1_score(y_val, (lr_proba>=0.5).astype(int))},
    {'Model': f'KNN (k={best_k})', 'Module': 8,
     'Val AUC': knn_auc,
     'Val F1': f1_score(y_val, (knn_proba>=0.5).astype(int))},
    {'Model': 'Decision Tree (D=4)', 'Module': '9/10',
     'Val AUC': roc_auc_score(y_val, dt_best.predict_proba(X_val_pp)[:,1]),
     'Val F1': f1_score(y_val, dt_best.predict(X_val_pp))},
    {'Model': 'Random Forest (200 trees)', 'Module': 10,
     'Val AUC': rf_val_auc, 'Val F1': rf_val_f1},
    {'Model': 'Gradient Boosting (200 est.)', 'Module': 10,
     'Val AUC': gb_val_auc, 'Val F1': gb_val_f1},
])
summary = summary.sort_values('Val AUC', ascending=False)
print('\nModel Performance Summary (Validation Set):')
print(summary.to_string(index=False))
print('\nGradient Boosting and Random Forest significantly outperform single models.')
print('This motivates XGBoost as the primary model under audit in the Audit Toolkit.')

---
## Next Notebook

→ **Notebook 04:** Naïve Bayes, Logistic Regression & SVM (Modules 11, 13, 14)